# Comparación remota de modelos con Weights & Biases

Este notebook reúne los entrenamientos de **ambas laptops directamente desde W&B**. No necesita los modelos, los audios, las carpetas de las suites ni una GPU. Consulta configuraciones, métricas por época y resúmenes; descarga únicamente las matrices de confusión y tablas por clase que se soliciten.

Compara LeNet-5 y MobileNetV1, con datos base y aumentados. El mejor modelo se elige por **macro-F1 de validación descendente**, luego **loss de validación ascendente**, nombre del experimento y ruta del run. Test se presenta como evaluación independiente, sin intervenir en la selección.

W&B solo permite recuperar lo que se haya subido: las ejecuciones `offline` deben sincronizarse desde la laptop que conserva sus registros. Las métricas de test que todavía no se calcularon no pueden reconstruirse a partir de las curvas de entrenamiento.

## 1. Dependencias y configuración

Use el entorno de `ai/requirements.txt` del repositorio. Para un entorno de análisis nuevo bastan las dependencias siguientes; descomente la instalación si las necesita. El notebook no importa PyTorch ni ejecuta entrenamiento.

In [1]:
# %pip install wandb==0.29.0 pandas==2.2.3 numpy==1.26.4 matplotlib==3.10.3 Pillow==11.2.1 nbformat==5.10.4

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import wandb
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'ai/tools/wandb_comparison.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abra este notebook dentro del repositorio OnnxMobile.')
AI = ROOT / 'ai'
OUTPUT = AI / 'runs' / 'wandb_comparison'
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('WANDB_CACHE_DIR', str(AI / '.cache/wandb'))
os.environ.setdefault('WANDB_DATA_DIR', str(AI / '.cache/wandb-data'))
sys.path.insert(0, str(AI / 'tools'))
from wandb_comparison import collect_runs, rank_runs, match_tests, download_media, read_class_table

# entity = usuario o equipo que aparece en https://wandb.ai/<entity>/<project>
# Si ambas laptops usaron el mismo proyecto, una sola entrada incluye ambas.
SOURCES = [
    {'entity': os.environ.get('WANDB_ENTITY', 'danielpulido-instituto-tecnol-gico-de-costa-rica'),
     'project': os.environ.get('WANDB_PROJECT', 'onnxmobile-speech-commands'),
     'groups': []},  # [] = todos los grupos; o ['suite_laptop_1', 'suite_laptop_2']
    # {'entity': 'otro-usuario-o-equipo', 'project': 'otro-proyecto', 'groups': []},
]
DATASET_FINGERPRINT = None  # Opcional: limitar a una exportación específica.
COHORT = None  # Si hay datasets/pérdidas diferentes, copie un cohort de la tabla del paso 3.
DOWNLOAD_MEDIA = True
PUBLISH_COMPARISON = False  # True crea un NUEVO run de comparación al ejecutar la última celda.
PUBLISH_ENTITY = SOURCES[0]['entity']
PUBLISH_PROJECT = SOURCES[0]['project']

# Los 12 experimentos de la configuración actual. Ajuste si su investigación usa otra grilla.
config = json.loads((AI / 'configs/experiments.json').read_text(encoding='utf-8'))
EXPECTED_EXPERIMENTS = [f"{architecture}-{variant}-{trial['name']}"
                        for architecture in ('lenet5', 'mobilenet_v1')
                        for variant in ('base', 'augmented') for trial in config['trials']]

## 2. Autenticación y consulta de las dos laptops

Complete `SOURCES` con las rutas de sus proyectos; puede usar distintos usuarios, equipos y grupos, siempre que su cuenta tenga acceso de lectura. `wandb.login()` reutiliza la sesión guardada o solicita la clave de forma interactiva; también puede definir `WANDB_API_KEY` en el entorno. **No escriba la clave en una celda ni la guarde en Git.**

Para resultados offline, ejecute en **cada laptop de origen**, desde la raíz del repositorio:

```powershell
Get-ChildItem -LiteralPath 'ai/runs/<su_suite>/wandb' -Directory -Filter 'offline-run-*' |
    ForEach-Object { wandb sync $_.FullName }
```

El comando sincroniza cada carpeta offline de la suite. Si hay varias suites, repita para cada una. Después vuelva a ejecutar la consulta. No cambie `entity`/`project` durante la sincronización para conservar los vínculos de los runs de test.

Los entrenamientos online ya registran las curvas en tiempo real en el workspace de W&B. Este notebook obtiene una **instantánea**; vuelva a ejecutar desde esta celda para actualizarla. Un run todavía activo aparece en el inventario, pero no participa en la elección.

In [ ]:
if any(not s['entity'] or not s['project'] for s in SOURCES):
    raise ValueError('Complete entity y project en SOURCES antes de conectar con W&B.')
wandb.login()
api = wandb.Api(timeout=60)
records, test_runs, histories, remote_runs, notices = collect_runs(api, SOURCES)
queried_at = datetime.now(timezone.utc).isoformat()
for notice in notices:
    warnings.warn(notice)
display(Markdown(f'Consultados **{len(records)} entrenamientos** y **{len(test_runs)} runs de test**.'))
display(records[['run_path', 'group', 'experiment', 'state', 'epochs', 'epochs_ran',
                 'eligible', 'exclusion_reason']])
records.to_csv(OUTPUT / 'runs_inventory.csv', index=False)

{'entity': 'danielpulido-instituto-tecnol-gico-de-costa-rica', 'project': 'onnxmobile-speech-commands', 'groups': []}


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\danie\_netrc.
wandb: Currently logged in as: danielpulido (danielpulido-instituto-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. Comparabilidad de los resultados

Una cohorte reúne runs con la **misma huella del dataset base**, definición de la pérdida y pesos de clase. La huella base identifica las mismas particiones de validación/test; la huella del entrenamiento puede variar entre datos base y aumentados. No se mezclan automáticamente datasets diferentes ni pérdidas con pesos distintos al desempatar.

Solo participan entrenamientos `finished`, con todas las épocas configuradas y métricas válidas del mejor checkpoint. Si falta procedencia o algún resultado, el inventario explica la exclusión. Un fallo al leer curvas se informa y no se sustituye por ceros.

Las métricas `validation/*` del resumen suelen ser las de la **última época**. La clasificación usa `best_validation_*`; las métricas adicionales se completan desde la fila de `best_epoch`, si está disponible.

In [ ]:
cohorts = (records.groupby(['cohort', 'dataset_fingerprint', 'loss_definition'], dropna=False)
           .agg(runs=('run_path', 'size'), eligible_runs=('eligible', 'sum')).reset_index())
display(cohorts)
display(records[['run_path', 'architecture', 'variant', 'training_dataset_fingerprint',
                 'source_sha256', 'host', 'loss_definition']])
ranking, winners = rank_runs(records, cohort=COHORT, dataset_fingerprint=DATASET_FINGERPRINT)
missing = sorted(set(EXPECTED_EXPERIMENTS) - set(ranking['experiment']))
expected_groups = {(a, v) for a in ('lenet5', 'mobilenet_v1') for v in ('base', 'augmented')}
missing_groups = sorted(expected_groups - set(zip(winners.architecture, winners.variant)))
provisional = bool(missing or missing_groups)
if provisional:
    warnings.warn(f'Comparación provisional. Experimentos pendientes/excluidos: {missing}; grupos faltantes: {missing_groups}')
duplicates = ranking[ranking.duplicated(['experiment', 'seed'], keep=False)]
if not duplicates.empty:
    display(Markdown('Hay repeticiones del mismo experimento/semilla. Se conservan con su ruta; no son semillas independientes.'))
    display(duplicates[['run_path', 'experiment', 'seed', 'best_validation_macro_f1']])

## 4. Ranking y mejor modelo

Macro-F1 da el mismo peso a cada clase y es el criterio principal de esta investigación. Accuracy, balanced accuracy, baseline mayoritario y brecha de generalización aportan contexto. Las diferencias pequeñas de una sola semilla no demuestran superioridad estadística; hacen falta repeticiones para estimar variabilidad.

Los ganadores por arquitectura y variante pueden tener hiperparámetros distintos. La diferencia entre ganador aumentado y base describe la receta completa, incluido su presupuesto de cómputo; no aísla el efecto del aumento.

In [ ]:
columns = ['rank', 'experiment', 'group', 'best_epoch', 'best_validation_macro_f1',
           'best_validation_accuracy', 'best_validation_balanced_accuracy', 'best_validation_loss',
           'macro_f1_gap', 'majority_validation_macro_f1', 'parameters', 'optimizer', 'lr',
           'batch_size', 'epochs', 'seed', 'url']
display(ranking.reindex(columns=columns))
display(Markdown('### Ganadores por arquitectura y variante'))
display(winners.reindex(columns=columns))
champion = ranking.iloc[0]
status = 'provisional entre los runs disponibles' if provisional else 'entre los experimentos comparados'
conclusion = (f"Mejor modelo **{status}**: **{champion['experiment']}**. "
              f"Macro-F1 de validación: **{champion['best_validation_macro_f1']:.4f}**; "
              f"loss: **{champion['best_validation_loss']:.4f}**; época: **{int(champion['best_epoch'])}**. "
              f"[Ver ejecución en W&B]({champion['url']}). La selección no utiliza test.")
display(Markdown(conclusion))

fig, ax = plt.subplots(figsize=(12, max(4, len(ranking) * .5)))
chart = ranking.set_index('run_path')[['best_validation_macro_f1', 'best_validation_accuracy']]
chart.rename(columns={'best_validation_macro_f1': 'Macro-F1', 'best_validation_accuracy': 'Accuracy'}).plot.barh(ax=ax)
ax.invert_yaxis()
ax.set(xlabel='Validación del mejor checkpoint', xlim=(0, 1), ylabel='Run')
fig.tight_layout()
fig.savefig(OUTPUT / 'ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Curvas de entrenamiento y señales de overfitting

Se muestran todas las épocas, sin el muestreo de `run.history()`. `train_clean` mide el entrenamiento original sin aumento ni muestreo, en modo evaluación, y permite comparar la brecha con validación. `train` refleja el proceso de optimización y puede incluir datos aumentados/muestreados.

Una brecha creciente y una pérdida de validación que sube mientras baja la de train limpio **sugieren sobreajuste**. La brecha por sí sola no es una prueba. Rendimiento bajo en ambos conjuntos debe interpretarse junto con el baseline y los errores por clase; no se etiqueta automáticamente como subajuste.

In [ ]:
curve_paths = []
for row in ranking.to_dict('records'):
    history = histories[row['run_path']]
    if history.empty:
        display(Markdown(f"Sin curvas disponibles: `{row['run_path']}`."))
        continue
    history.to_csv(OUTPUT / f"history_{row['run_path'].replace('/', '__')}.csv", index=False)
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    for ax, metric in zip(axes.flat[:3], ('loss', 'accuracy', 'macro_f1')):
        for split, label in [('train', 'Train (optimización)'), ('train_clean', 'Train limpio'), ('validation', 'Validación')]:
            column = f'{split}/{metric}'
            if column in history and history[column].notna().any():
                ax.plot(history.epoch, history[column], label=label)
        ax.axvline(row['best_epoch'], color='gray', linestyle=':', label='Mejor época')
        ax.set(xlabel='época', ylabel=metric)
        ax.legend(fontsize=8)
    ax = axes.flat[3]
    if {'train_clean/macro_f1', 'validation/macro_f1'} <= set(history):
        ax.plot(history.epoch, history['train_clean/macro_f1'] - history['validation/macro_f1'], label='Train limpio − validación')
        ax.legend(fontsize=8)
    else:
        ax.text(.5, .5, 'No se registró train_clean', ha='center', transform=ax.transAxes)
    ax.axhline(0, color='gray', linestyle=':')
    ax.set(xlabel='época', ylabel='Brecha de macro-F1')
    fig.suptitle(f"{row['experiment']} | {row['run_path']}")
    fig.tight_layout()
    target = OUTPUT / f"curves_{row['run_path'].replace('/', '__')}.png"
    fig.savefig(target, dpi=150, bbox_inches='tight')
    curve_paths.append(target)
    plt.show()

display(ranking[['run_path', 'best_epoch', 'macro_f1_gap', 'last_macro_f1_gap',
                 'tail_validation_loss_change', 'tail_train_clean_loss_change']])
# Los cambios de loss usan hasta las últimas tres épocas, no una prueba estadística.

## 6. Test ya registrado en W&B

Se vincula cada evaluación al `training_run_id`, al proyecto, a la huella del dataset y al SHA-256 del checkpoint. No se unen resultados solo por nombre: dos laptops pueden usar el mismo nombre de experimento. Se conservan las evaluaciones repetidas como filas distintas.

Si no aparece test, la comparación por validación sigue siendo posible. Para obtener test hace falta evaluarlo en la laptop que tenga los pesos y datos (flujo del notebook 02), y sincronizar su run; este notebook no lo inventa ni lo ejecuta.

In [ ]:
test_results, test_notices = match_tests(ranking, test_runs)
for notice in test_notices:
    warnings.warn(notice)
if test_results.empty:
    display(Markdown('No hay resultados de test compatibles registrados para esta selección de runs.'))
else:
    display(test_results)
    selected_tests = test_results[test_results['training_run_path'].isin(winners['run_path'])]
    display(Markdown('### Test de los ganadores seleccionados con validación'))
    display(selected_tests)

## 7. Matrices de confusión y métricas por clase

Se consultan `validation_best/confusion` y `validation_best/per_class` de los ganadores, y `test/confusion` y `test/per_class` de sus evaluaciones compatibles. Las matrices originales tienen filas de clase real y columnas de predicción. Las tablas muestran precision, recall, F1 y support por clase.

Solo se descargan archivos de media. Si un archivo no se subió, fue eliminado o no tiene una ruta recuperable, se informa y se conserva el enlace al run para inspeccionarlo en W&B.

In [ ]:
media_paths, class_tables = [], []
media_targets = [(path, 'validation_best') for path in winners['run_path']]
if not test_results.empty:
    media_targets += [(r['test_run_path'], 'test') for r in test_results.to_dict('records')
                      if r['training_run_path'] in set(winners['run_path'])]
if DOWNLOAD_MEDIA:
    for run_path, prefix in media_targets:
        run = remote_runs[run_path]
        display(Markdown(f"### [{run_path}]({run.url}) — {prefix}"))
        for kind in ('confusion', 'per_class'):
            try:
                path = download_media(run, f'{prefix}/{kind}', OUTPUT / 'media')
                if path is None:
                    display(Markdown(f'No hay archivo recuperable de `{prefix}/{kind}`. Revise el enlace al run.'))
                    continue
                media_paths.append(path)
                if kind == 'confusion':
                    display(Image(filename=str(path)))
                else:
                    table = read_class_table(path)
                    class_tables.append((run_path, prefix, table))
                    display(table)
                    table.to_csv(OUTPUT / f"classes_{run_path.replace('/', '__')}_{prefix}.csv", index=False)
            except Exception as error:
                warnings.warn(f'{run_path}, {prefix}/{kind}: no se pudo recuperar ({type(error).__name__}). Revise el run en W&B.')
else:
    display(Markdown('Descarga de matrices y tablas desactivada con DOWNLOAD_MEDIA=False.'))

## 8. Tamaño, cómputo y límites de la comparación

Parámetros, tamaño del estado y MACs Conv/Linear describen el costo del modelo. La latencia PyTorch y el tiempo de entrenamiento proceden de **laptops posiblemente diferentes**: no se usan para elegir al ganador ni para afirmar que una arquitectura será más rápida en Android. Para decidir el despliegue hay que medir ONNX, preprocesamiento y consumo en el mismo teléfono.

Las fuentes de código quedan visibles en el paso 3. Si difieren, revise si cambiaron las arquitecturas o el protocolo de evaluación. Este análisis describe los runs disponibles; una sola semilla no estima incertidumbre.

In [ ]:
display(winners[['run_path', 'parameters', 'conv_linear_macs', 'state_bytes',
                 'training_samples', 'samples_per_epoch', 'optimizer_steps',
                 'training_seconds', 'forward_median_ms', 'profile_device', 'host']])

## 9. Exportar evidencia y, opcionalmente, registrar la comparación en W&B

Se guarda el inventario, ranking, ganadores, test, historiales, figuras y un informe Markdown en `ai/runs/wandb_comparison`. No contiene pesos ni claves de API.

Con `PUBLISH_COMPARISON=True`, la última celda crea un **nuevo run** `job_type="comparison"` con tablas, figuras y un artefacto del informe. Las ejecuciones originales se consultan en modo lectura. Después de editar las fuentes o filtros, vuelva a ejecutar el notebook desde la consulta para evitar mezclar instantáneas.

In [ ]:
ranking.to_csv(OUTPUT / 'ranking.csv', index=False)
winners.to_csv(OUTPUT / 'winners.csv', index=False)
test_results.to_csv(OUTPUT / 'test_results.csv', index=False)
selection = {
    'queried_at_utc': queried_at, 'sources': SOURCES,
    'cohort': champion['cohort'], 'dataset_fingerprint': champion['dataset_fingerprint'],
    'criterion': 'validation macro-F1 desc, loss asc, experiment asc, run_path asc; test never used',
    'provisional': provisional, 'missing_experiments': missing, 'missing_groups': missing_groups,
    'best_overall_run_path': champion['run_path'], 'best_overall_url': champion['url'],
    'winner_run_paths': winners['run_path'].tolist(), 'notices': notices + test_notices,
}
(OUTPUT / 'selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding='utf-8')
lines = ['# Comparación remota con W&B', '', conclusion, '',
         f'Consulta UTC: {queried_at}. Cohorte: {champion["cohort"]}.', '',
         '| Experimento | Run | Macro-F1 validación | Accuracy validación | Loss validación | época |',
         '| --- | --- | ---: | ---: | ---: | ---: |']
for r in ranking.to_dict('records'):
    accuracy = r.get('best_validation_accuracy')
    accuracy_text = f'{accuracy:.4f}' if pd.notna(accuracy) else 'No registrado'
    lines.append(f"| {r['experiment']} | [{r['run_path']}]({r['url']}) | {r['best_validation_macro_f1']:.4f} | {accuracy_text} | {r['best_validation_loss']:.4f} | {int(r['best_epoch'])} |")
lines += ['', f'Experimentos faltantes o excluidos: {missing}.', '',
          'El ranking se fijó por validación. test_results.csv contiene solo evaluaciones ya registradas y compatibles.',
          'Las curvas y matrices se exportan cuando están disponibles. Los valores ausentes no se sustituyen por ceros.',
          'Una sola semilla no demuestra superioridad estadística. Latencias de laptops diferentes no son comparables.',
          'El aumento puede cambiar el número de pasos de optimización; los ganadores pueden usar distintos hiperparámetros.',
          'No se descargaron checkpoints ni se evaluó test nuevamente.', '']
(OUTPUT / 'report.md').write_text('\n'.join(lines), encoding='utf-8')
display(Markdown(f'Archivos exportados en `{OUTPUT}`.'))

In [ ]:
if PUBLISH_COMPARISON:
    if not PUBLISH_ENTITY or not PUBLISH_PROJECT:
        raise ValueError('Complete PUBLISH_ENTITY y PUBLISH_PROJECT.')
    with wandb.init(entity=PUBLISH_ENTITY, project=PUBLISH_PROJECT, mode='online',
                    dir=str(OUTPUT), job_type='comparison',
                    name='comparacion-remota-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'),
                    config=selection) as comparison:
        comparison.log({'ranking': wandb.Table(dataframe=ranking),
                        'winners': wandb.Table(dataframe=winners),
                        'ranking_chart': wandb.Image(str(OUTPUT / 'ranking.png'))})
        if not test_results.empty:
            comparison.log({'registered_test_results': wandb.Table(dataframe=test_results)})
        for index, path in enumerate(curve_paths):
            comparison.log({f'curves/{index}': wandb.Image(str(path))})
        for index, path in enumerate(p for p in media_paths if p.suffix.lower() == '.png'):
            comparison.log({f'confusion/{index}': wandb.Image(str(path))})
        for index, (run_path, prefix, table) in enumerate(class_tables):
            comparison.log({f'per_class/{index}': wandb.Table(dataframe=table)})
        comparison.summary.update({'best_overall_run_path': champion['run_path'],
                                   'best_validation_macro_f1': champion['best_validation_macro_f1'],
                                   'provisional': provisional, 'n_training_runs': len(ranking)})
        artifact = wandb.Artifact(f'comparison-{comparison.id}', type='report')
        for filename in ('runs_inventory.csv', 'ranking.csv', 'winners.csv', 'test_results.csv', 'selection.json', 'report.md'):
            artifact.add_file(str(OUTPUT / filename), name=filename)
        comparison.log_artifact(artifact)
        display(Markdown(f'[Abrir comparación en W&B]({comparison.url})'))
else:
    display(Markdown('Publicación desactivada. Active PUBLISH_COMPARISON para subir el informe a W&B.'))

## Referencias

- [W&B Public API: consultar y exportar runs e historiales](https://docs.wandb.ai/models/track/public-api-guide).
- [Referencia de Run: scan_history y métricas registradas](https://docs.wandb.ai/models/ref/python/public-api/run).
- [Descarga de archivos registrados](https://docs.wandb.ai/models/ref/python/public-api/file).
- [Sincronización de ejecuciones offline](https://docs.wandb.ai/models/ref/cli/wandb-sync).

El notebook compara la evidencia registrada. Para volver a entrenar o evaluar pesos, use el notebook 02 y el snapshot de código correspondiente a cada suite.